# Baseline - Standalone MPNet

This notebook performs evaluation on the base MPNet model without further training.

The input data for this notebook is embedded data output from the `Text Embedding` notebook.

# Set up

In [2]:
import numpy as np
import pandas as pd
# from util import *

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score

def f1(gpt_dist, answer_dist, threshold):
    # positives = GPT-vs-GPT pairs
    # negatives = GPT-vs-human pairs
    scores = np.concatenate([gpt_dist, answer_dist])

    # 1 = AI-like pair, 0 = human-like pair
    y_true = np.concatenate([
        np.ones(len(gpt_dist), dtype=int),
        np.zeros(len(answer_dist), dtype=int)
    ])

    # smaller distance => more likely AI-like
    y_pred = (scores <= threshold).astype(int)

    precision = precision_score(y_true, y_pred, zero_division=0)
    recall = recall_score(y_true, y_pred, zero_division=0)
    f1_val = f1_score(y_true, y_pred, zero_division=0)

    return precision, recall, f1_val

def run_model(answers, gpt1, gpt2, seeds=(1111,2222,3333,4444,5555,6666,7777,8888,9999,1010)):
  valid_accuracies = []
  valid_f1s = []
  test_accuracies = []
  test_f1s = []


  for seed in seeds:
    np.random.seed(seed)

    #train-test-validation split
    sampling_indices = np.random.uniform(0,1,answers.shape[0])

    valid_answers = answers[(0.8 <= sampling_indices) & (sampling_indices < 0.9)]
    valid_gpt1 = gpt1[(0.8 <= sampling_indices) & (sampling_indices < 0.9)]
    valid_gpt2 = gpt2[(0.8 <= sampling_indices) & (sampling_indices < 0.9)]

    test_answers = answers[sampling_indices >= 0.9]
    test_gpt1 = gpt1[sampling_indices >= 0.9]
    test_gpt2 = gpt2[sampling_indices >= 0.9]

    #valid accuracy
    valid_gpt_distances_org = ((valid_gpt1 - valid_gpt2)**2).sum(axis=1)
    valid_answer_distances_org = ((valid_gpt1 - valid_answers)**2).sum(axis=1)
    valid_accuracies.append((valid_gpt_distances_org < valid_answer_distances_org).mean())

    #tune threshold
    thresholds = []
    perf = []
    for thres in np.arange(0,0.5,0.01):
      thresholds.append(thres)
      perf.append(f1(valid_gpt_distances_org, valid_answer_distances_org, thres)[2])

    emb_thres = thresholds[perf.index(max(perf))]
    emb_thres, max(perf)

    #valid f1
    valid_f1s.append(max(perf))

    #test accuracy
    test_gpt_distances_org = ((test_gpt1 - test_gpt2)**2).sum(axis=1)
    test_answer_distances_org = ((test_gpt1 - test_answers)**2).sum(axis=1)
    test_accuracies.append((test_gpt_distances_org < test_answer_distances_org).mean())

    #test f1
    test_f1s.append(f1(test_gpt_distances_org, test_answer_distances_org, emb_thres)[2])
  return valid_accuracies, test_accuracies, valid_f1s, test_f1s

## Load Data

We need to load the embeddings for answer, gpt1, and gpt2. Change `dataset` to use the data of interests.

In [5]:
dataset = 'SQUAD_GPT4'

import pickle

with open(f'/home/aanis/Metric-Models-for-Detection-of-LLM-Texts/Data/{dataset}_answer_embs.obj', 'rb') as f:
  answers = pickle.load(f)
with open(f'/home/aanis/Metric-Models-for-Detection-of-LLM-Texts/Data/{dataset}_gpt1_embs.obj', 'rb') as f:
  gpt1 = pickle.load(f)
with open(f'/home/aanis/Metric-Models-for-Detection-of-LLM-Texts/Data/{dataset}_gpt2_embs.obj', 'rb') as f:
  gpt2 = pickle.load(f)

# Evaluation

Run the function to evaluate the base model

In [7]:
print(type(answers))
print(type(gpt1))
print(type(gpt2))

print(getattr(answers, "shape", None))
print(getattr(gpt1, "shape", None))
print(getattr(gpt2, "shape", None))

<class 'pandas.Series'>
<class 'pandas.Series'>
<class 'pandas.Series'>
(4306,)
(4306,)
(4306,)


In [8]:
print(type(answers.iloc[0]))
print(type(gpt1.iloc[0]))
print(type(gpt2.iloc[0]))

<class 'numpy.ndarray'>
<class 'numpy.ndarray'>
<class 'numpy.ndarray'>


In [45]:
import numpy as np

answers_np = np.stack(answers.to_list())
gpt1_np = np.stack(gpt1.to_list())
gpt2_np = np.stack(gpt2.to_list())

valid_accuracies, test_accuracies, valid_f1s, test_f1s = run_model(
    answers_np, gpt1_np, gpt2_np
)

In [ ]:
# valid_accuracies, test_accuracies, valid_f1s, test_f1s = run_model(answers,gpt1,gpt2)

Get the average performance

In [46]:
np.mean(valid_accuracies), np.mean(test_accuracies), np.mean(valid_f1s), np.mean(test_f1s)

(np.float64(0.9823191131063036),
 np.float64(0.9803060560306601),
 np.float64(0.8753116834471711),
 np.float64(0.869386000397202))

In [21]:
print(type(valid_accuracies))
print(len(valid_accuracies))
print(type(valid_accuracies[0]))
print(valid_accuracies[0])

<class 'list'>
10
<class 'numpy.float64'>
0.981651376146789


In [22]:
for i, x in enumerate(valid_accuracies[:3]):
    print(i, type(x), x)

0 <class 'numpy.float64'> 0.981651376146789
1 <class 'numpy.float64'> 0.9848484848484849
2 <class 'numpy.float64'> 0.9763593380614657
